# Synthetic vibration, order and envelope-analysis walkthrough

This notebook exercises the complete public toolkit on deterministic synthetic signals. Frequencies before resampling are in Hz; frequencies after position resampling are in cycles per integrated speed unit (order when speed is in rev/s). FFT values are coherent-gain-corrected **amplitudes**, not PSD estimates.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from signal_analyzer.demodulation import amplitude_demodulate
from signal_analyzer.fft_tools import (
    averaged_fft,
    averaged_psd,
    fft_spectrum,
    power_spectrum,
    psd_spectrum,
)
from signal_analyzer.file_utils import find_files
from signal_analyzer.indicators import (
    amplitude_band_statistics,
    coordinate_range_mask,
    psd_band_statistics,
    signal_statistics,
)
from signal_analyzer.resampling import (
    max_speed_aa_filter,
    speed_resample,
    speed_resample_aa,
)
from signal_analyzer.sampling import resample_uniform
from signal_analyzer.synthetic_signals import (
    simulate_bearing_fault,
    synthesize_tone_mix,
)

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

plt.rcParams.update({"figure.figsize": (9, 3.5), "axes.grid": True})

## 1. Configurable temporal and order components

The first generator adds fixed temporal tones and shaft-order components. The smooth speed profile spans 2.5–10 rev/s: a 4:1 ratio compatible with the default 4× spatial grid under the documented grid-density check.

In [ ]:
fs = 2048.0
duration = 4.0

def speed_profile(t):
    return 6.25 + 3.75 * np.sin(2 * np.pi * t / duration)

t, speed, position, x = synthesize_tone_mix(
    fs=fs,
    duration=duration,
    speed=speed_profile,
    temporal_components=((120.0, 0.35, 0.0), (310.0, 0.15, 0.4)),
    order_components=((1.0, 0.8, 0.0), (3.0, 0.45, 0.2), (5.0, 0.2, 0.7)),
    noise_std=0.05,
    seed=7,
)

fig, axes = plt.subplots(2, 1, sharex=True)
axes[0].plot(t, speed)
axes[0].set_ylabel("speed [rev/s]")
axes[1].plot(t, x, lw=0.8)
axes[1].set(xlim=(0, 0.5), xlabel="time [s]", ylabel="x [amplitude]")
plt.tight_layout()

## 2. Temporal FFT and blockwise amplitude averaging

`fft_spectrum` retains complex phase. `averaged_fft` averages block magnitudes—not complex FFTs and not power/PSD. Its incomplete trailing block is omitted.

In [ ]:
f, Xf = fft_spectrum(x, fs=fs, window="hann")
f_avg, amplitude_avg, block_centers, S = averaged_fft(
    x, 1024, fs=fs, window="hann", overlap=0.5,
    return_spectrogram=True,
)

fig, ax = plt.subplots()
ax.plot(f, np.abs(Xf), alpha=0.45, label="full-record amplitude")
ax.plot(f_avg, amplitude_avg, label="mean block magnitude")
ax.set(xlim=(0, 400), xlabel="frequency [Hz]", ylabel="amplitude")
ax.legend()
print(f"spectrogram: {S.shape[0]} blocks × {S.shape[1]} bins")

## 3. PSD, power-per-bin, indicators, and uniform downsampling

PSD uses window-power density normalization, while the amplitude spectrum above uses coherent-gain correction. Power-per-bin is PSD multiplied by bin width; summing it estimates mean square. Band RMS is calculated only from PSD, never from the averaged amplitude spectrum. Coordinate ranges are half-open.

In [ ]:
f_psd, psd = psd_spectrum(x, fs=fs, window="hann")
f_psd_avg, psd_avg = averaged_psd(
    x, 1024, fs=fs, window="hann", overlap=0.5
)
f_power, power_per_bin = power_spectrum(x, fs=fs, window="hann")

time_zone = coordinate_range_mask(t, lower=0.5, upper=1.5)
time_stats = signal_statistics(x[time_zone], sample_spacing=1 / fs)
amplitude_stats = amplitude_band_statistics(
    f_avg, amplitude_avg, lower=110, upper=130
)
psd_stats = psd_band_statistics(f_psd_avg, psd_avg, lower=110, upper=130)

x_down, fs_down = resample_uniform(x, fs=fs, down=2)

print(f"time-zone RMS: {time_stats['rms']:.3f}")
print(f"120 Hz amplitude peak: {amplitude_stats['peak_amplitude']:.3f}")
print(f"110–130 Hz band RMS from PSD: {psd_stats['band_rms']:.3f}")
print(f"mean square / summed power bins: {np.mean(x**2):.3f} / {power_per_bin.sum():.3f}")
print(f"uniform downsample: {len(x)} @ {fs:g} Hz -> {len(x_down)} @ {fs_down:g} Hz")

fig, ax = plt.subplots()
ax.semilogy(f_psd, psd, alpha=0.4, label="full-record PSD")
ax.semilogy(f_psd_avg, psd_avg, label="averaged PSD")
ax.set(xlim=(0, 400), xlabel="frequency [Hz]", ylabel="PSD [amplitude²/Hz]")
ax.legend()

## 4. Temporal anti-alias filter and position resampling

For target resolution 64 samples/rev, algorithmic spatial Nyquist is 32 orders. The practical working band used here ends at `64 / 2.56 = 25` orders. The max-speed temporal cutoff is `max(speed) × resolution / 2`; it is permissive and is not a proof of alias-free nonuniform resampling.

In [ ]:
resolution = 64
practical_order_limit = resolution / 2.56
temporal_cutoff = speed.max() * resolution / 2

x_prefiltered = max_speed_aa_filter(x, speed, fs, resolution=resolution)
pos_direct, x_direct = speed_resample(x, speed, fs, resolution=resolution)
pos_aa, x_aa = speed_resample_aa(x, speed, fs, resolution=resolution)

print(f"temporal cutoff: {temporal_cutoff:.1f} Hz")
print(f"prefilter RMS change: {np.sqrt(np.mean((x - x_prefiltered)**2)):.3f}")
print(f"practical order limit: {practical_order_limit:.1f}")
print(f"direct/AA samples: {len(x_direct)} / {len(x_aa)}")

fig, ax = plt.subplots()
ax.plot(pos_direct, x_direct, alpha=0.5, label="direct interpolation")
ax.plot(pos_aa, x_aa, lw=1, label="filtered + 4× + decimated")
ax.set(xlim=(2, 4), xlabel="position [rev]", ylabel="amplitude")
ax.legend()

## 5. Spatial spectrum and wavelength view

With 64 samples/rev, the FFT axis is cycles/rev (order). Wavelength is its reciprocal in rev/cycle. The zero-frequency bin must be removed, and reciprocal coordinates are nonuniform and reverse the original frequency ordering.

In [ ]:
order, order_amplitude = averaged_fft(
    x_aa, 512, fs=resolution, window="hann", overlap=0.5
)

positive = order > 0
wavelength = 1 / order[positive]
wavelength_amplitude = order_amplitude[positive]
wavelength_order = np.argsort(wavelength)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(order, order_amplitude)
axes[0].axvline(practical_order_limit, color="C3", ls="--", label="resolution / 2.56")
axes[0].set(xlim=(0, practical_order_limit), xlabel="order [cycles/rev]", ylabel="amplitude")
axes[0].legend()
axes[1].plot(wavelength[wavelength_order], wavelength_amplitude[wavelength_order])
axes[1].set(xscale="log", xlim=(0.04, 2), xlabel="wavelength [rev/cycle]", ylabel="amplitude")
plt.tight_layout()

## 6. Variable-speed motor with bearing-like impacts

The second generator combines shaft orders 1–5, fixed temporal tones, noise, and fault-order impacts that excite a decaying high-frequency resonance. This is a generic diagnostic example, not a bearing or machine-specific physical model.

In [ ]:
bearing_fs = 4096.0
fault_order = 4.8
t_b, speed_b, position_b, x_b = simulate_bearing_fault(
    fs=bearing_fs,
    duration=4.0,
    speed_min=2.5,
    speed_max=10.0,
    speed_cycles=1.0,
    shaft_order_amplitudes=(0.6, 0.3, 0.2, 0.12, 0.08),
    constant_components=((180.0, 0.15, 0.0), (320.0, 0.08, 0.4)),
    fault_order=fault_order,
    fault_amplitude=1.0,
    resonance_frequency=900.0,
    resonance_decay=0.004,
    noise_std=0.03,
    seed=12,
)

fig, axes = plt.subplots(2, 1, sharex=True)
axes[0].plot(t_b, speed_b)
axes[0].set_ylabel("speed [rev/s]")
axes[1].plot(t_b, x_b, lw=0.7)
axes[1].set(xlim=(0, 0.5), xlabel="time [s]", ylabel="x [amplitude]")
plt.tight_layout()

In [ ]:
f_b, amplitude_b = averaged_fft(
    x_b, 2048, fs=bearing_fs, window="hann", overlap=0.5
)
fig, ax = plt.subplots()
ax.plot(f_b, amplitude_b)
ax.axvspan(700, 1100, color="C1", alpha=0.15, label="demodulation band")
ax.set(xlim=(0, 1400), xlabel="frequency [Hz]", ylabel="mean magnitude")
ax.legend()

## 7. Demodulate before spatial resampling

The carrier band is specified in Hz, so demodulation is performed on the original time grid. Hilbert output retains amplitude units; square-law output has squared units. Forward–backward filtering has no causal delay but does have endpoint transients, so 0.25 s is trimmed from both ends before resampling.

In [ ]:
demod_kwargs = dict(
    fs=bearing_fs, band=(700.0, 1100.0), lowpass=100.0, remove_mean=True
)
envelope_hilbert = amplitude_demodulate(x_b, detector="hilbert", **demod_kwargs)
envelope_square = amplitude_demodulate(x_b, detector="square", **demod_kwargs)

edge = int(0.25 * bearing_fs)
keep = slice(edge, -edge)
pos_h, env_h = speed_resample_aa(
    envelope_hilbert[keep], speed_b[keep], bearing_fs, resolution=resolution
)
pos_s, env_s = speed_resample_aa(
    envelope_square[keep], speed_b[keep], bearing_fs, resolution=resolution
)

order_h, amplitude_h = averaged_fft(
    env_h, 512, fs=resolution, window="hann", overlap=0.5
)
order_s, amplitude_s = averaged_fft(
    env_s, 512, fs=resolution, window="hann", overlap=0.5
)

fig, ax = plt.subplots()
ax.plot(order_h, amplitude_h, label="Hilbert envelope [amplitude]")
ax.plot(order_s, amplitude_s, alpha=0.75, label="square law [amplitude²]")
for shaft_order in range(1, 6):
    ax.axvline(shaft_order, color="0.75", lw=0.8)
ax.axvline(fault_order, color="C3", ls="--", label=f"fault order {fault_order:g}")
ax.set(xlim=(0, 12), xlabel="order [cycles/rev]", ylabel="demodulated magnitude")
ax.legend()

## 8. Recursive file discovery

`find_files` returns `pathlib.Path` objects sorted case-insensitively by filename rather than full path.

In [ ]:
python_files = find_files(repo_root, "*.py")
[path.relative_to(repo_root) for path in python_files[:8]]

## Takeaways

- Temporal spectra use Hz; spatial spectra use cycles per integrated unit.
- Averaged spectra are mean amplitudes, not PSDs.
- The `/2.56` band is a practical working limit inside the algorithmic Nyquist boundary.
- Max-speed filtering plus 4× spatial oversampling is a partial safeguard for the demonstrated 4:1 speed range, not a rigorous nonuniform reconstruction guarantee.
- Demodulation belongs before resampling when its carrier band is defined in Hz. Trim zero-phase filter endpoints before amplitude-sensitive work.
- Wavelength is reciprocal frequency: omit zero and remember that its sample spacing is nonuniform.